In [ ]:
dagdelen_per_week = 3 # @param {"type":"slider","min":0,"max":8,"step":1}
extra_uren_per_week = 2 # @param {"type":"number"}
aantal_weken = 40 # @param {"type":"number"}
herkomst_jongere = "Groningen" # @param ["Groningen","Drenthe"]
max_aantal_productcodes = "5" # @param ["1","2","3","4","5"]
max_aantal_productcodes = int(max_aantal_productcodes)

MBO3_percentage = 0 # @param {"type":"slider","min":0,"max":1,"step":0.05}
MBO4_percentage = 0.2 # @param {"type":"slider","min":0,"max":1,"step":0.05}
HBO_plus_percentage = 0.2 # @param {"type":"slider","min":0,"max":1,"step":0.05}
WO_percentage = 0.4 # @param {"type":"slider","min":0,"max":1,"step":0.05}
WO_plus_percentage = 0.1 # @param {"type":"slider","min":0,"max":1,"step":0.05}

gewenste_verdeling = {
    'MBO3': MBO3_percentage,
    'MBO4': MBO4_percentage,
    'HBO+': HBO_plus_percentage,
    'WO': WO_percentage,
    'WO+': WO_plus_percentage
}

# Gegevens
CAO_bruto_salaris = {
    '10-8': 2571,
    '20-8': 2715,
    '30-8': 2990,
    '40-8': 3272,
    '50-8': 4100,
    '60-8': 5307,
    '70-8': 7288,
    '80-8': 9868
}
ORT_percentage = 0.01
Sociale_lasten_percentage = 0.2619
Niet_declarabel_percentage = 0.32
Reiskosten_per_uur = 1.5 # In euro's
Overhead_percentage = 0.22
Marge_percentage = 0.02
werkuren_per_maand = 156.5
dertiende_maand = 1/12*13

# Berekeningen per schaal/trede
resultaten = []

for schaal, bruto in CAO_bruto_salaris.items():
#    dertiende_maand_bruto = dertiende_maand * bruto
#    bruto = bruto + dertiende_maand_bruto
    ORT = bruto * ORT_percentage
    sociale_lasten = bruto * Sociale_lasten_percentage
    niet_declarabel = bruto * Niet_declarabel_percentage
    reiskosten_per_maand = Reiskosten_per_uur * werkuren_per_maand

    # Totaal zonder overhead en marge
    totaal_zonder_overhead = bruto + ORT + sociale_lasten + niet_declarabel + reiskosten_per_maand

    # Overhead en marge
    overhead = totaal_zonder_overhead * Overhead_percentage
    marge = (totaal_zonder_overhead + overhead) * Marge_percentage

    # Totale maandelijkse kosten
    totale_kosten = totaal_zonder_overhead + overhead + marge

    # Uurtarief en kosten per minuut
    uurtarief = totale_kosten / werkuren_per_maand
    kosten_per_minuut = uurtarief / 60

    resultaten.append({
        'Schaal + trede': schaal,
        'Bruto salaris': bruto,
        'ORT': ORT,
        'Sociale lasten': sociale_lasten,
        'Niet-declarabel': niet_declarabel,
        'Reiskosten': reiskosten_per_maand,
        'Overhead': overhead,
        'Marge': marge,
        'Totale maandelijkse kosten': totale_kosten,
        'Uurtarief': uurtarief,
        'Kosten per minuut': kosten_per_minuut
    })

# Resultaten afdrukken
# for result in resultaten:
  #   print(f"Schaal + trede: {result['Schaal + trede']}")
  #   print(f"Kosten per minuut: {result['Kosten per minuut']:.2f} €\n")

# Een boekje aanmaken waarin de kosten  per minuut voor iedere schaal + trede staan
resultaten_dict = {result['Schaal + trede']: result for result in resultaten}


import numpy as np
from scipy.optimize import linprog

# =============================================
# INVOER: Kosten per opleidingsniveau (per minuut)
# =============================================
# Bruto salaris + ORT + sociale lasten + overhead + risico/marge (per minuut)
kosten_per_minuut = {
    'MBO3': resultaten_dict['10-8']['Kosten per minuut'],
    'MBO4': resultaten_dict['20-8']['Kosten per minuut'],
    'HBO+': resultaten_dict['30-8']['Kosten per minuut'],
    'WO': resultaten_dict['40-8']['Kosten per minuut'],
    'WO+': resultaten_dict['50-8']['Kosten per minuut'],
    'SKJ': 0.10      # Voor SKJ-geregistreerd
}

# =============================================
# INVOER: Tarief per minuut (per productcode) (ondernemingsplan en RIGG, geindexeerd met 5.13%)
# =============================================
tarief_per_minuut = {
    # Drenthe
    '45A63': 1.01,   # Begeleiding licht
    '45A04': 1.17,   # Begeleiding midden
    '45A70': 1.28,   # Begeleiding zwaar
    '41A22': 0.29,   # Dagbesteding basis
    '45A23': 0.36,   # Dagbesteding intensief
    '54001': 1.81,   # GGZ basis
    '54002': 1.99,   # GGZ specialistisch
    '45A49': 1.35,   # Gezinsbehandeling licht, ambulant
    '45A51': 1.35,   # Gezinsbehandeling, echtscheidingsproblematiek

    # Groningen
    '52G09': 137.20 / 60,   # Individuele behandeling specialistisch
    '45A53': 109.50 / 60,   # Individuele begeleiding specialistisch
    '45A56': 109.50 / 60,   # Gezinsbegeleiding specialistisch
    '45G02': 135.90 / 60,   # Gezinsdiagnostiek
    '45G03': 119.20 / 60,   # Intensieve ambulante gezinsbehandeling
    '45G04': 148.25 / 60,   # Systeemtherapie
    '45G06': 126.15 / 60    # Dagbesteding
    }

max_aanvraag = {
    # Drenthe
    '45A63': 6000,   # Begeleiding licht
    '45A04': 6000,   # Begeleiding midden
    '45A70': 6000,   # Begeleiding zwaar
    '41A22': 6000,   # Dagbesteding basis
    '45A23': 6000,   # Dagbesteding intensief
    '54001': 6000,   # GGZ basis
    '54002': 6000,   # GGZ specialistisch
    '45A49': 6000,   # Gezinsbehandeling licht, ambulant
    '45A51': 6000,    # Gezinsbehandeling, echtscheidingsproblematiek

    # Groningen
    '52G09': 6000,   # Individuele behandeling specialistisch
    '45A53': 6000,   # Individuele begeleiding specialistisch
    '45A56': 6000,   # Gezinsbegeleiding specialistisch
    '45G02': 6000,   # Gezinsdiagnostiek
    '45G03': 6000,   # Intensieve ambulante gezinsbehandeling
    '45G04': 6000,   # Systeemtherapie
    '45G06': 6000    # Dagbesteding
}
max_aanvraag = np.array([[code, value] for code, value in max_aanvraag.items()])

# =============================================
# INVOER: Verhoudingen per productcode
# =============================================
verhoudingen = {
    # Drenthe
    '45A63': {'MBO3': 0.50, 'MBO4': 0.50},  # Begeleiding licht
    '45A04': {'MBO4': 0.80, 'HBO+': 0.20},  # Begeleiding midden
    '45A70': {'MBO4': 0.35, 'HBO+': 0.60, 'WO': 0.05},  # Begeleiding zwaar
    '41A22': {'MBO3': 0.55, 'MBO4': 0.42, 'WO+': 0.03},  # Dagbesteding basis
    '45A23': {'MBO3': 0.35, 'MBO4': 0.40, 'HBO+': 0.20, 'WO+': 0.05},  # Dagbesteding intensief
    '54001': {'HBO+': 0.50, 'WO': 0.35, 'WO+': 0.15},  # GGZ basis
    '54002': {'HBO+': 0.45, 'WO': 0.40, 'WO+': 0.15},  # GGZ specialistisch
    '45A49': {'MBO3': 0.50, 'MBO4': 0.50},  # Gezinsbehandeling licht, ambulant
    '45A51': {'MBO3': 0.50, 'MBO4': 0.50},  # Gezinsbehandeling, echtscheidingsproblematiek

    # Groningen
    '52G09': {'MBO3': 0.50, 'MBO4': 0.50},  # Individuele behandeling specialistisch
    '45A53': {'MBO4': 0.80, 'HBO+': 0.20},  # Individuele begeleiding specialistisch
    '45A56': {'MBO4': 0.35, 'HBO+': 0.60, 'WO': 0.05},  # Gezinsbegeleiding specialistisch
    '45G02': {'MBO3': 0.55, 'MBO4': 0.42, 'WO+': 0.03},  # Gezinsdiagnostiek
    '45G03': {'MBO3': 0.35, 'MBO4': 0.40, 'HBO+': 0.20, 'WO+': 0.05},  # Intensieve ambulante gezinsbehandeling
    '45G04': {'HBO+': 0.50, 'WO': 0.35, 'WO+': 0.15},  # Systeemtherapie
    '45G06': {'HBO+': 0.45, 'WO': 0.40, 'WO+': 0.15},  # Dagbesteding
}

# =============================================
# Functie om dagdelen om te zetten naar minuten per jaar
# =============================================
def dagdelen_naar_minuten_per_jaar(dagdelen_per_week, aantal_weken):
    uren_per_dagdeel = 4
    minuten_per_uur = 60

    uren_per_week = dagdelen_per_week * uren_per_dagdeel
    uren_per_jaar = uren_per_week * aantal_weken
    minuten_per_jaar = uren_per_jaar * minuten_per_uur

    return minuten_per_jaar



# =============================================
# INVOER: Actieve productcodes en uren
# =============================================

# actieve_productcodes_drenthe = ['45A63', '45A04', '45A70', '41A22', '45A23', '54001', '54002', '45A49', '45A51']
actieve_productcodes_drenthe = ['45A63', '45A04', '45A70', '41A22', '45A23', '54001', '54002', '45A49', '45A51']

# actieve_productcodes_groningen = ['52G09', '45A53', '45A56', '45G02', '45G03', '45G04', '45G06']
actieve_productcodes_groningen = ['52G09', '45A53', '45A56', '45G02', '45G03', '45G04', '45G06']

# =============================================
# Totaal beschikbare minuten (dagbesteding + extra uren)
# =============================================
totaal_minuten_dagbesteding = dagdelen_naar_minuten_per_jaar(dagdelen_per_week, aantal_weken)
totaal_minuten_extra = extra_uren_per_week * aantal_weken * 60
totaal_minuten = totaal_minuten_dagbesteding + totaal_minuten_extra


import numpy as np
from scipy.optimize import linprog

# =============================================
# Parameter: Alpha bepaalt de balans tussen winst en verdeling
# =============================================
alpha_values = np.linspace(0, 1, 100)  # 11 waarden tussen 0 en 1

if herkomst_jongere == 'Groningen':
  actieve_productcodes = actieve_productcodes_groningen
elif herkomst_jongere == 'Drenthe':
  actieve_productcodes = actieve_productcodes_drenthe

# Variabelen om de beste resultaten op te slaan
beste_winst = -float('inf')
beste_error_factor = float('inf')
beste_resultaten = None

for alpha in alpha_values:
    # =============================================
    # Stap 1: Doelstellingsfunctie opstellen (winst + verdeling)
    # =============================================
    # Winstmaximalisatie
    c_winst = [
        -(tarief_per_minuut[code] - sum(kosten_per_minuut[niveau] * verhoudingen[code].get(niveau, 0) for niveau in gewenste_verdeling))
        for code in actieve_productcodes
    ]
    # Verdelingsdoel: minimaliseer afwijking van gewenste verdeling
    c_verdeling = np.zeros(len(actieve_productcodes))
    for i, code in enumerate(actieve_productcodes):
        for niveau in gewenste_verdeling:
            ratio = verhoudingen[code].get(niveau, 0)
            c_verdeling[i] += ratio * (0 - gewenste_verdeling[niveau])  # Streef naar gewenste verdeling
    # Combineer winst en verdeling
    c_combined = (
        alpha * np.array(c_winst) +
        (1 - alpha) * c_verdeling
    )

    # =============================================
    # Stap 2: Constraints voor totale minuten en verdeling
    # =============================================
    A_eq = [[1] * len(actieve_productcodes)]  # Totaal aantal minuten
    b_eq = [totaal_minuten]
    # Constraints voor minimale/maximale verdeling per niveau
    A_ub = []
    b_ub = []
    for niveau in gewenste_verdeling:
        # Minimale minuten voor dit niveau (bijv. 70% van gewenst)
        min_minuten = gewenste_verdeling[niveau] * totaal_minuten * 0.5
        row_min = [-verhoudingen[code].get(niveau, 0) for code in actieve_productcodes]
        A_ub.append(row_min)
        b_ub.append(-min_minuten)
        # Maximale minuten voor dit niveau (bijv. 130% van gewenst)
        max_minuten = gewenste_verdeling[niveau] * totaal_minuten * 1.5
        row_max = [verhoudingen[code].get(niveau, 0) for code in actieve_productcodes]
        A_ub.append(row_max)
        b_ub.append(max_minuten)
    bounds = [(0, None) for _ in actieve_productcodes]

    # =============================================
    # Stap 3: Optimalisatie uitvoeren
    # =============================================
    result = linprog(c_verdeling, A_eq=A_eq, b_eq=b_eq, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    if not result.success:
        print(f"Geen optimale oplossing gevonden met alpha={alpha:.1f} en strikte constraints. Probeer met losse constraints...")
        # Probeer zonder verdelingsconstraints
        result = linprog(c_verdeling, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
        if not result.success:
            print(f"Geen haalbare oplossing gevonden voor alpha={alpha:.1f}. Controleer de inputgegevens.")
            continue

    # Rond af op veelvouden van 10
    optimale_minuten = np.round(result.x / 10) * 10
    optimale_minuten = optimale_minuten.astype(int)

    # Zorg dat de som gelijk is aan totaal_minuten
    if sum(optimale_minuten) > totaal_minuten:
        idx = np.argmax(optimale_minuten)
        optimale_minuten[idx] -= 10
    elif sum(optimale_minuten) < totaal_minuten:
        idx = np.argmin(optimale_minuten)
        optimale_minuten[idx] += 10

    # =============================================
    # Selecteer alleen de top 'max_aantal_productcodes' productcodes
    # =============================================
    top_indices = np.argsort(optimale_minuten)[-max_aantal_productcodes:]
    optimale_minuten_top = np.zeros_like(optimale_minuten)
    optimale_minuten_top[top_indices] = optimale_minuten[top_indices]

    # Zorg dat de som gelijk blijft aan totaal_minuten
    som_top = sum(optimale_minuten_top)
    if som_top > 0:
        optimale_minuten_top[top_indices] = (optimale_minuten_top[top_indices] / som_top) * totaal_minuten
    optimale_minuten_top = np.round(optimale_minuten_top / 100) * 100
    optimale_minuten = optimale_minuten_top.astype(int)

    # =============================================
    # Stap 4: Bereken huidige verdeling, omzet, kosten, winst
    # =============================================
    huidige_verdeling = {niveau: 0 for niveau in gewenste_verdeling}
    for i, code in enumerate(actieve_productcodes):
        for niveau in gewenste_verdeling:
            huidige_verdeling[niveau] += optimale_minuten[i] * verhoudingen[code].get(niveau, 0)

    optimale_omzet = sum(tarief_per_minuut[code] * optimale_minuten[i] for i, code in enumerate(actieve_productcodes))
    optimale_kosten = sum(
        sum(kosten_per_minuut[niveau] * verhoudingen[code].get(niveau, 0) for niveau in gewenste_verdeling) * optimale_minuten[i]
        for i, code in enumerate(actieve_productcodes)
    )
    optimale_winst = optimale_omzet - optimale_kosten

    # =============================================
    # Stap 5: Bereken error factor
    # =============================================
    error_factor = 0
    for niveau in gewenste_verdeling:
        huidige_percent = (huidige_verdeling[niveau] / totaal_minuten) * 100
        gewenst_percent = gewenste_verdeling[niveau] * 100
        error_factor += abs(huidige_percent - gewenst_percent)
    error_factor /= len(gewenste_verdeling)  # Gemiddelde absolute afwijking per niveau

    # =============================================
    # Bewaar de beste resultaten: maximale winst, minimale error
    # =============================================
    if (optimale_winst > beste_winst) or (optimale_winst == beste_winst and error_factor < beste_error_factor):
        beste_winst = optimale_winst
        beste_error_factor = error_factor
        beste_resultaten = {
            'alpha': alpha,
            'optimale_minuten': optimale_minuten,
            'huidige_verdeling': huidige_verdeling,
            'error_factor': error_factor,
            'omzet': optimale_omzet,
            'kosten': optimale_kosten,
            'winst': optimale_winst
        }

# =============================================
# Toon de beste resultaten
# =============================================
print("=== Beste resultaten (maximale winst, minimale error) ===")
print(f"Beste alpha: {beste_resultaten['alpha']:.2f} (Totale winst: €{beste_resultaten['winst']:.2f}, Error factor: {beste_resultaten['error_factor']:.2f}%)")
print("\nOptimale verdeling (afgerond op 10 minuten, maximaal 3 productcodes):")
for i, code in enumerate(actieve_productcodes):
    if beste_resultaten['optimale_minuten'][i] > 0:
        print(f"- {code}: {beste_resultaten['optimale_minuten'][i]} minuten")
        for niveau in gewenste_verdeling:
            if verhoudingen[code].get(niveau, 0) > 0:
                print(f"  - {niveau}: {beste_resultaten['optimale_minuten'][i] * verhoudingen[code][niveau]:.2f} minuten")
print(f"\nOptimale omzet: €{beste_resultaten['omzet']:.2f}")
print(f"Optimale kosten: €{beste_resultaten['kosten']:.2f}")
print(f"Totale winst: €{beste_resultaten['winst']:.2f}")
print("\nHuidige verdeling vs. gewenst:")
for niveau in gewenste_verdeling:
    print(f"- {niveau}: {beste_resultaten['huidige_verdeling'][niveau]/totaal_minuten:.0%} (gewenst: {gewenste_verdeling[niveau]:.0%})")
print(f"\nError factor (gemiddelde absolute afwijking per niveau): {beste_resultaten['error_factor']:.2f}%")


=== Beste resultaten (maximale winst, minimale error) ===
Beste alpha: 0.00 (Totale winst: €58900.48, Error factor: 2.00%)

Optimale verdeling (afgerond op 10 minuten, maximaal 3 productcodes):
- 45G04: 33600 minuten
  - HBO+: 16800.00 minuten
  - WO: 11760.00 minuten
  - WO+: 5040.00 minuten

Optimale omzet: €83020.00
Optimale kosten: €24119.52
Totale winst: €58900.48

Huidige verdeling vs. gewenst:
- MBO3: 0% (gewenst: 0%)
- MBO4: 0% (gewenst: 0%)
- HBO+: 50% (gewenst: 50%)
- WO: 35% (gewenst: 40%)
- WO+: 15% (gewenst: 10%)

Error factor (gemiddelde absolute afwijking per niveau): 2.00%
